# Неделя 2 — Сборка обучающей выборки

Карточка недели: `docs/week2_dataset.md` · Полное описание: тетрадь, раздел 6

**Конвенция ячеек:**
- ячейка без пометки — готовый код, просто запустите
- `# TODO:` — здесь пишете вы

## 1. Определение таргета

TODO: зафиксируйте здесь четыре пункта (объект наблюдения, событие оттока, окна, обоснование).
Шаблон — тетрада 6.3, шаг 1. Продублируйте в журнале.

In [1]:
import sys

import numpy as np
import pandas as pd

sys.path.append('..')
from src.features import (FORECAST_MONTHS, GAP_MONTHS, OBSERVATION_MONTHS,
                          SNAPSHOTS, build_snapshot, window_bounds)

clients = pd.read_parquet('../data/processed/clients.parquet')
usage = pd.read_parquet('../data/processed/usage.parquet')
print(clients.shape, usage.shape)

(50400, 8) (1073079, 7)


## 2. Проверка границ окон

Прежде чем собирать выборку, убедитесь, что `window_bounds` считает то, что нужно.

In [2]:
print(window_bounds('2025-01'))
# ожидаем ('2024-08', '2025-01', '2025-04', '2025-06')

(Period('2024-08', 'M'), Period('2025-01', 'M'), Period('2025-04', 'M'), Period('2025-06', 'M'))


## 3. Один snapshot — глазами

Соберите ОДИН снапшот и посмотрите на него, прежде чем запускать цикл по семи.

In [3]:
# TODO: реализуйте make_features(), make_target(), build_snapshot(), затем:
one = build_snapshot(usage, clients, '2025-01')
print(one.shape)
print('доля оттока:', one['target'].mean().round(4))
one.head(20)

(35788, 27)
доля оттока: 0.0842


,client_id,rev_mean,rev_min,rev_max,rev_last,traffic_mean,traffic_last,sim_last,sim_mean,sum_tickets,...,rev_trend,traffic_trend,rev_trend_norm,rev_months_declining,sim_change,target,segment,product,region,snapshot
0,CL000001,1.819404e+06,1753016.27,1889117.87,1753016.27,1718.133333,1507.3,74,70.000000,0,...,-28857.501714,-23.211429,-0.015861,5,0,0,крупный бизнес,телефония,Юг,2025-01
1,CL000002,2.009294e+06,1904798.92,2102392.81,1926903.97,2024.933333,1509.2,104,107.500000,1,...,14274.025429,-1.091429,0.007104,2,-9,0,крупный бизнес,телефония,Юг,2025-01
2,CL000004,1.096042e+05,104496.60,116423.51,107036.62,120.200000,112.3,17,16.500000,0,...,-1719.306571,-5.588571,-0.015687,1,1,0,средний бизнес,облачные сервисы,Сибирь,2025-01
3,CL000006,1.358691e+05,121663.35,150270.21,121663.35,134.100000,105.4,23,23.833333,0,...,-5723.291714,-9.245714,-0.042124,5,-1,0,средний бизнес,телефония,Москва,2025-01
4,CL000007,2.295501e+05,199371.75,248463.70,238875.28,209.983333,247.7,10,9.833333,2,...,7457.920286,7.380000,0.032489,1,0,0,средний бизнес,интернет,Юг,2025-01
5,CL000008,4.303366e+05,373252.06,458550.61,448919.80,412.150000,512.0,45,49.166667,3,...,-8019.146857,19.454286,-0.018635,0,-2,0,крупный бизнес,облачные сервисы,Урал,2025-01
6,CL000010,5.351594e+04,2079.37,98022.72,2079.37,50.066667,2.1,8,13.500000,5,...,-20574.908000,-16.605714,-0.384463,5,-12,1,средний бизнес,мобильная связь,Москва,2025-01
7,CL000011,3.953851e+05,272123.22,576622.76,444314.76,390.575000,478.7,234,248.750000,1,...,80471.764000,96.990000,0.203528,1,-18,1,крупный бизнес,облачные сервисы,Сибирь,2025-01
8,CL000012,6.213862e+04,58545.76,65731.49,58545.76,65.050000,46.2,10,9.500000,0,...,-7185.730000,-37.700000,-0.115640,1,1,0,средний бизнес,телефония,Юг,2025-01
9,CL000013,1.061114e+05,99106.30,112517.06,112517.06,116.466667,94.1,23,22.333333,3,...,1791.458857,-7.074286,0.016883,0,1,0,средний бизнес,облачные сервисы,Сибирь,2025-01


## 4. Сборка по всем снапшотам

In [4]:
dataset = pd.concat([build_snapshot(usage, clients, s) for s in SNAPSHOTS],
                    ignore_index=True)
dataset.to_parquet('../data/processed/dataset.parquet')
print(dataset.shape)

(257631, 27)


## 5. Обязательные проверки

Ожидаемая картина: доля оттока по снапшотам меняется плавно.
Медленный дрейф — норма. Скачки в разы — баг.

In [5]:
print(dataset.groupby('snapshot')['target'].agg(['mean', 'count']).round(4))

            mean  count
snapshot               
2024-07   0.0645  32156
2024-10   0.0720  34097
2025-01   0.0842  35788
2025-04   0.0960  37297
2025-07   0.1037  38564
2025-10   0.1135  39464
2026-01   0.1258  40265


In [6]:
assert dataset['segment'].isna().sum() == 0, 'потеряли сегмент при merge'
assert not dataset.duplicated(subset=['client_id', 'snapshot']).any(), 'дубли клиент x снапшот'
print('проверки пройдены')

проверки пройдены


## 6. Самопроверка на утечку

TODO: таблица по каждому признаку — из каких месяцев считается и мог ли я знать
это значение в день snapshot. Признак с ответом «нет» или «не уверен» в модель не идёт.

| Признак | Из каких месяцев | Мог ли знать в день snapshot? |
|---|---|---|
|  |  |  |